# SimJEB 3 - evaluation

Scores the trained model on the held-out test set. **Run this once, at the end.**
Every look at the test set during development turns it into a second validation set,
and model selection slowly overfits to it.

**Settings:** Accelerator **GPU T4**, Internet **ON**. Add notebook 1's output and
notebook 2's output as input datasets.

Three things get reported, and the last two are what make the first mean anything:
aggregate metrics **in MPa**, a **bootstrap interval**, and a **trivial baseline**.

In [ ]:
GITHUB_REPO = "https://github.com/Vedavamsi-3/simjeb-structural-gnn.git"
DATA  = "/kaggle/input/simjeb-data"      # notebook 1 output
TRAIN = "/kaggle/input/simjeb-train"     # notebook 2 output
RUN   = "C"

REPO = "/kaggle/working/simjeb-structural-gnn"

import subprocess, sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

if GITHUB_REPO and not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-geometric", "meshio", "trimesh"], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)

import numpy as np, pandas as pd, torch
from IPython.display import Image, display

OUT = Path("/kaggle/working/outputs")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Kaggle names mounted datasets itself, so locate them rather than hard-coding paths.
_data = sorted(Path("/kaggle/input").glob("*/graphs"))
if _data:
    DATA = str(_data[0].parent)
    print("data:", DATA)

_ckpt = sorted(Path("/kaggle/input").glob("*/outputs/*/best_model.pt"))
if _ckpt:
    TRAIN = str(_ckpt[0].parents[2])
    RUN = _ckpt[0].parent.name
    print("training output:", TRAIN, "| run:", RUN)

CHECKPOINT = Path(TRAIN) / "outputs" / RUN / "best_model.pt"
if not CHECKPOINT.exists():
    print("available under /kaggle/input:", [str(x) for x in Path("/kaggle/input").glob("*")])
assert CHECKPOINT.exists(), f"no checkpoint at {CHECKPOINT}"
print("checkpoint:", CHECKPOINT)


## The training curve it came from

In [ ]:
history = pd.read_csv(Path(TRAIN) / "outputs" / RUN / "history.csv")
print(f"epochs {len(history)} | best val R2 {history.val_r2_mpa.max():.4f} "
      f"at epoch {history.val_r2_mpa.idxmax()}")
display(Image(str(Path(TRAIN) / "outputs" / RUN / f"loss_curve_{RUN}.png")))

## Test set

Scored on the **grouped** split -- no design family straddles train and test. The
official split is scored underneath for comparison; the gap between the two is what
same-family overlap was worth.

In [ ]:
from src.evaluate import evaluate, plot_per_model_r2, plot_prediction_vs_truth

report = evaluate(
    checkpoint_path=CHECKPOINT,
    graph_dir=f"{DATA}/graphs",
    split_path=f"{DATA}/splits/grouped_split_v1.json",
    run_name=RUN,
    device="cuda",
    out_dir=str(OUT),
)

### Reading the numbers

- **R² in MPa**, not log space. R² on a log target flatters the model, because the log
  compresses exactly the large errors that matter.
- **The 95% interval** comes from resampling whole *models*, not nodes. Nodes within a
  bracket are highly correlated; resampling them would treat millions of dependent
  points as independent and produce an interval far too narrow to be honest.
- **The trivial baseline** predicts the training mean at every node. It should score
  ~0. If it scores well, the task or the split is too easy and the headline number is
  meaningless.

In [ ]:
flagged = set()
flagged_csv = Path(DATA) / "qa" / "flagged.csv"
if flagged_csv.exists():
    flagged = set(pd.read_csv(flagged_csv)["model_id"].tolist())

plot_per_model_r2(report, FIG / "per_model_r2.png", flagged=flagged)
plot_prediction_vs_truth(report, CHECKPOINT, f"{DATA}/graphs",
                         f"{DATA}/splits/grouped_split_v1.json",
                         FIG / "prediction_vs_truth.png", device="cuda")
display(Image(str(FIG / "per_model_r2.png")))
display(Image(str(FIG / "prediction_vs_truth.png")))

**What to look for in the scatter.** A fan opening at high stress means the singular
peaks are the hard part -- the expected failure mode, worth confirming rather than
assuming. Systematic curvature would mean bias instead.

**In the per-model bars**, whether the orange (QA-flagged) models cluster in the left
tail. If they do, the QA stage predicted where the model would struggle.

In [ ]:
per_model = pd.DataFrame([vars(m) for m in report.per_model])
per_model["qa_flagged"] = per_model.model_id.isin(flagged)
per_model.to_csv(OUT / "per_model_results.csv", index=False)

print("worst five models:")
print(per_model.head(5)[["model_id", "r2", "mae_mpa", "max_true_mpa", "qa_flagged"]]
      .to_string(index=False))
print("\nbest five:")
print(per_model.tail(5)[["model_id", "r2", "mae_mpa", "max_true_mpa", "qa_flagged"]]
      .to_string(index=False))

if flagged:
    print(f"\nmedian R2 -- flagged {per_model[per_model.qa_flagged].r2.median():.3f} "
          f"vs clean {per_model[~per_model.qa_flagged].r2.median():.3f}")

## The same model on the official split

For comparability with published work, and to quantify what design-family overlap was
worth. Same weights, same test-set size -- only the split membership differs.

In [ ]:
official_path = Path(DATA) / "splits" / "official_split_0.json"
official_report = None
if official_path.exists():
    official_report = evaluate(
        checkpoint_path=CHECKPOINT, graph_dir=f"{DATA}/graphs",
        split_path=str(official_path), run_name=f"{RUN}_official",
        device="cuda", out_dir=str(OUT),
    )
    print(f"\ngrouped  R2 {report.r2:.4f}   CI [{report.r2_ci95[0]:.3f}, {report.r2_ci95[1]:.3f}]")
    print(f"official R2 {official_report.r2:.4f}   CI [{official_report.r2_ci95[0]:.3f}, {official_report.r2_ci95[1]:.3f}]")
    print(f"gap        {official_report.r2 - report.r2:+.4f}")

**Caveat on that comparison.** The model was trained on the *grouped* split, so some
of the official split's test models were in its training set. The official number is
therefore optimistic for two reasons at once, and is reported for context rather than
as a clean benchmark. A like-for-like comparison would need a second training run on
the official split.

## Summary

In [ ]:
import json

summary = {
    "run": RUN,
    "split": report.split_name,
    "test_models": report.n_models,
    "r2": round(report.r2, 4),
    "r2_ci95": [round(v, 4) for v in report.r2_ci95],
    "mae_mpa": round(report.mae_mpa, 1),
    "rmse_mpa": round(report.rmse_mpa, 1),
    "baseline_r2": round(report.baseline_r2, 4),
    "epochs_trained": len(history),
    "official_split_r2": round(official_report.r2, 4) if official_report else None,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))

print(f"von Mises surface stress, vertical load case, {report.n_models} unseen brackets")
print(f"  R2    {report.r2:.4f}   95% CI [{report.r2_ci95[0]:.4f}, {report.r2_ci95[1]:.4f}]")
print(f"  MAE   {report.mae_mpa:.1f} MPa")
print(f"  RMSE  {report.rmse_mpa:.1f} MPa")
print(f"  trivial baseline R2: {report.baseline_r2:.4f}")
print("\nfor the README:")
print(json.dumps(summary, indent=2))